# DeBERTa grid on Kaggle — the three cells Colab did not finish

`{1e-5, 3}` is already done and committed. Remaining: **`{1e-5, 5}`, `{2e-5, 3}`,
`{2e-5, 5}`** — 15 runs, roughly 105 minutes, inside Kaggle's 12-hour session limit.

**Two things to set before running, both in the right-hand sidebar:**

1. **Session options → Accelerator → GPU T4 x2** (or P100)
2. **Session options → Internet → On** — off by default, and without it the clone
   and the HuggingFace download both fail. Needs a phone-verified account.

Unlike Colab there is no `files.download`. Anything written to `/kaggle/working`
is kept as the notebook's Output and downloadable from there, so each cell's
results are copied out the moment it finishes — a dropped session costs one cell.

The repo and the 73 MB corpus are cloned to `/tmp`, not `/kaggle/working`, to keep
them out of the output.


In [ ]:
# 1. Kaggle guard, internet, GPU, and the command helper.
import os, sys, socket, subprocess, pathlib

if not pathlib.Path('/kaggle').is_dir():
    raise SystemExit('This notebook is for Kaggle. On Colab use colab_deberta_grid.ipynb.')

try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError as e:
    raise SystemExit(
        f'No internet ({e}). Kaggle disables it by default.\n'
        'Sidebar -> Session options -> Internet -> On (needs a phone-verified account).')

import torch
assert torch.cuda.is_available(), 'No GPU. Sidebar -> Session options -> Accelerator -> GPU.'
print(torch.cuda.get_device_name(0), '|', torch.__version__, '| cuda', torch.version.cuda)

WORK = pathlib.Path('/kaggle/working')        # persisted as notebook Output
REPO = pathlib.Path('/tmp/ate-acter')         # scratch: repo + corpus stay out of Output

def run(*args, cwd=None):
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=cwd)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {" ".join(str(a) for a in args)}')


In [ ]:
# 2. Clone the project and the corpus into /tmp.
import shutil
shutil.rmtree(REPO, ignore_errors=True)
run('git', 'clone', '-q', 'https://github.com/ahmedwaleedaref/ATE-ACTER.git', REPO)
run('git', 'clone', '-q', 'https://github.com/AylaRT/ACTER.git', REPO / 'data/raw/ACTER')
run('git', 'checkout', '-q', 'f05b09e985cad37eeaa8daa8b3f383197aa5324e',
    cwd=REPO / 'data/raw/ACTER')

log = subprocess.run(['git','log','--oneline','-80'], cwd=REPO,
                     capture_output=True, text=True).stdout
for needle, why in (('--model override', 'push 179a678'),
                    ('forces fp32', 'push 57f1b78, or DeBERTa will NaN'),
                    ('Cell folder is deberta_lr', 'push 2f7ab83, or --group names will not match --grid')):
    assert needle in log, f'clone predates "{needle}": {why}'
assert (REPO / 'data/raw/ACTER/en/htfl/annotated').is_dir(), 'ACTER checkout looks wrong'
print(subprocess.run(['git','log','--oneline','-1'], cwd=REPO,
                     capture_output=True, text=True).stdout)


In [ ]:
# 3. Pinned installs. torch/numpy are Kaggle's -- forcing them breaks its CUDA
#    build, exactly as on Colab. The run JSON records the torch version.
run(sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.16.1', 'tokenizers==0.23.1', 'safetensors==0.8.0',
    'huggingface_hub==1.29.0', 'sentencepiece==0.2.2', 'protobuf==7.36.0',
    'PyYAML==6.0.3', 'pytest==8.3.2')

# seqeval only builds under older setuptools; it is not on the training path.
def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a],
                          capture_output=True, text=True).returncode == 0
HAVE_SEQEVAL = (_pip('--no-build-isolation', 'seqeval==1.2.2')
                or (_pip('setuptools<81', 'wheel')
                    and _pip('--no-build-isolation', 'seqeval==1.2.2')))
print('seqeval:', 'installed (47 tests)' if HAVE_SEQEVAL else
      'UNAVAILABLE -- 46 of 47 tests; training unaffected')


In [ ]:
# 4. Environment check. Any gap from local is a confound for E05.
import importlib
for m in ('torch', 'transformers', 'tokenizers', 'numpy'):
    print(f'{m:14} {importlib.import_module(m).__version__}')
print('python', sys.version.split()[0],
      ' (local: 3.14.4 / torch 2.14.0; Colab ran {1e-5,3} on torch 2.11.0)')
print()
skip = [] if HAVE_SEQEVAL else ['--ignore=tests/test_seqeval_agreement.py']
run(sys.executable, '-m', 'pytest', 'tests/', '-q', *skip, cwd=REPO)


In [ ]:
# 5. The three remaining cells. Results copied to /kaggle/working after each.
import json
SEEDS = (42, 43, 44, 45, 46)
TODO = [('1e-5', 5), ('2e-5', 3), ('2e-5', 5)]

for lr, ep in TODO:
    group = f'deberta_grid/deberta_lr{float(lr):g}_e{ep}'
    src = REPO / 'results/runs' / group
    print(f'\n########## LR {lr}, {ep} epochs ##########')
    for s in SEEDS:
        run(sys.executable, '-m', 'src.models.run_train',
            '--model', 'microsoft/deberta-v3-base',
            '--lr', lr, '--epochs', ep,
            '--group', group, '--seed', s,
            '--reason', f'deberta grid lr={lr} epochs={ep}', cwd=REPO)
        r = json.loads((src / f'seed_{s}.json').read_text())
        print(f"    SEED {s}: best_epoch={r['best_epoch']} equi={r['best_equi_f1']:.4f} "
              f"htfl={r['htfl_f1']:.4f} collapsed={r['collapsed']} {r['wall_time_sec']}s")
    dest = WORK / group
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(src, dest)
    print(f'  -> copied to {dest} (persists as notebook Output)')


In [ ]:
# 6. Optional: {3e-5, 3}, the last cell of a full 3 x 2 grid. ~25 min.
#    {3e-5, 5} is E04 and is not rerun here.
lr, ep = '3e-5', 3
group = f'deberta_grid/deberta_lr{float(lr):g}_e{ep}'
src = REPO / 'results/runs' / group
for s in SEEDS:
    run(sys.executable, '-m', 'src.models.run_train',
        '--model', 'microsoft/deberta-v3-base', '--lr', lr, '--epochs', ep,
        '--group', group, '--seed', s,
        '--reason', f'deberta grid lr={lr} epochs={ep}', cwd=REPO)
    r = json.loads((src / f'seed_{s}.json').read_text())
    print(f"    SEED {s}: best_epoch={r['best_epoch']} equi={r['best_equi_f1']:.4f} "
          f"htfl={r['htfl_f1']:.4f} {r['wall_time_sec']}s")
dest = WORK / group
shutil.rmtree(dest, ignore_errors=True); shutil.copytree(src, dest)
print(f'  -> copied to {dest}')


In [ ]:
# 7. What to download: everything under /kaggle/working/deberta_grid.
#    Drop these folders into results/runs/deberta_grid/ locally, then run
#    `python -m src.aggregate --grid deberta` and `--compare deberta`.
for p in sorted((WORK / 'deberta_grid').rglob('seed_*.json')):
    print(' ', p.relative_to(WORK), f'{p.stat().st_size/1024:.0f} KB')
